In [1]:
!pip install z3-solver transformers pandas tqdm accelerate

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from z3 import *
import io
import contextlib
from tqdm import tqdm
import os
import gc

# =========================================================
# 1. LOAD DATASET
# =========================================================
input_file = '/kaggle/input/datasets/alirezaebrahimi/folio-h1-neuro-symbolic-data/folio_h1_experiment_v2.csv'
df = pd.read_csv(input_file)
print(f"Dataset loaded successfully with {len(df)} samples.", flush=True)

OUTPUT_DIR = '/kaggle/working/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. LOAD LOCAL MODEL AS SEMANTIC PARSER
# =========================================================
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"\nLoading Local Semantic Parser: {MODEL_ID} onto GPUs...", flush=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    device_map="auto", 
    torch_dtype=torch.bfloat16
)
print("Model loaded into VRAM successfully!\n", flush=True)

# =========================================================
# 3. Z3 PROMPT TEMPLATE
# =========================================================
Z3_PROMPT_TEMPLATE = """You are an automated First-Order Logic semantic parser.
Translate the natural language premises and conclusion into an executable Python script using the z3-solver library.

Strict Rules:
1. Import Z3: `from z3 import *`
2. Declare Sorts, Functions, and Constants.
3. Initialize solver: `s = Solver()`
4. Add all premises to the solver: `s.add(...)`
5. Add the NEGATION of the conclusion (Proof by Contradiction): `s.add(Not(...))`
6. Print EXACTLY 'UNSAT' if s.check() == unsat, or 'SAT' if s.check() == sat.
7. Return ONLY clean, executable Python code inside ```python ``` blocks. No extra explanations.

Premises:
{premises}

Conclusion:
{conclusion}"""

def translate_nl_to_z3_code(premises, conclusion):
    prompt = Z3_PROMPT_TEMPLATE.format(premises=premises, conclusion=conclusion)
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=400, do_sample=False)
        
    generated_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    
    if "```python" in generated_text:
        code = generated_text.split("```python")[1].split("```")[0].strip()
    elif "```" in generated_text:
        code = generated_text.split("```")[1].split("```")[0].strip()
    else:
        code = generated_text.strip()
        
    return code

# =========================================================
# 4. Z3 EXECUTION ENGINE
# =========================================================
def execute_z3_code(code_string):
    if not code_string:
        return "ERROR_EMPTY"
    
    output_buffer = io.StringIO()
    with contextlib.redirect_stdout(output_buffer):
        try:
            exec(code_string, globals())
            result = output_buffer.getvalue().strip()
            if 'UNSAT' in result.upper(): return 'UNSAT'
            elif 'SAT' in result.upper(): return 'SAT'
            else: return 'ERROR_LOGIC'
        except Exception:
            return 'ERROR_SYNTAX'

# =========================================================
# 5. AUTOMATED VERIFICATION PIPELINE (60 SAMPLES)
# =========================================================
SAMPLE_LIMIT = 60
df_subset = df.head(SAMPLE_LIMIT).copy()

results = []
successful_pairs = 0

print(f"Executing Local Neuro-Symbolic Verification on {len(df_subset)} problem instances...")

for index, row in tqdm(df_subset.iterrows(), total=len(df_subset), desc="Formal Proof Verification"):
    
    # 1. Process Original Text
    code_orig = translate_nl_to_z3_code(row['premises'], row['conclusion'])
    res_orig = execute_z3_code(code_orig)
    
    # 2. Process Caroline Text
    code_caro = translate_nl_to_z3_code(row['caroline_premises'], row['caroline_conclusion'])
    res_caro = execute_z3_code(code_caro)
    
    # FIXED LOGICAL CONCORDANCE CHECK:
    # Matches if both are UNSAT or both are SAT (formal equivalence preserved)
    is_match = (res_orig == res_caro) and (res_orig in ['SAT', 'UNSAT'])
    if is_match:
        successful_pairs += 1
        
    results.append({
        'id': row['id'],
        'z3_result_original': res_orig,
        'z3_result_caroline': res_caro,
        'is_perfect_match': is_match
    })

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()

# =========================================================
# 6. EXPORT RESULTS
# =========================================================
df_results = pd.DataFrame(results)
output_path = os.path.join(OUTPUT_DIR, 'z3_automated_verification_results.csv')
df_results.to_csv(output_path, index=False)

print("\n" + "="*55, flush=True)
print("AUTOMATED LOCAL Z3 VERIFICATION SUMMARY")
print("="*55, flush=True)
print(f"Semantic Parser: {MODEL_ID}")
print(f"Total Evaluated Sample Subset: {len(df_subset)}")
print(f"Formal Proof Concordance: N = {successful_pairs}")
print("="*55, flush=True)
print(f"Results saved to: {output_path}", flush=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.1/33.1 MB 54.5 MB/s eta 0:00:00:00:0100:01
Dataset loaded successfully with 204 samples.

Loading Local Semantic Parser: Qwen/Qwen2.5-7B-Instruct onto GPUs...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded into VRAM successfully!

Executing Local Neuro-Symbolic Verification on 60 problem instances...


Formal Proof Verification: 100%|██████████| 60/60 [43:49<00:00, 43.82s/it]



AUTOMATED LOCAL Z3 VERIFICATION SUMMARY
Semantic Parser: Qwen/Qwen2.5-7B-Instruct
Total Evaluated Sample Subset: 60
Formal Proof Concordance: N = 7
Results saved to: /kaggle/working/data/z3_automated_verification_results.csv
